In [6]:
!pip install --upgrade langchain langchain-openai langchain-huggingface langchain_community faiss-cpu tiktoken langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [langchain]


In [17]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

if "OPENAI_API_KEY" in os.environ:
    print("OpenAI API Key loaded successfully!")
else:
    print("ERROR: Could not find key. Make sure you set it in Colab Secrets (🔑).")

OpenAI API Key loaded successfully!


In [18]:
%%writefile policy.txt
Company Policy Manual

1. Work From Home (WFH) Policy
Our WFH policy allows employees to work remotely two (2) days per week. The designated remote days must be approved by your direct manager. All remote employees are expected to be available online during core business hours (10:00 AM - 4:00 PM).

2. Expense Policy
Employees can be reimbursed for pre-approved expenses. All expense reports must be submitted through the "FinancePort" system within 30 days of the purchase. Receipts are required for all expenses over $25. The company standard for meals while traveling is a maximum of $75 per day.

3. Vacation and Sick Leave
Full-time employees receive 15 days of paid time off (PTO) per year. This accrues bi-weekly. In addition, employees receive 5 paid sick days per year. Sick days do not roll over to the next year, but up to 5 unused PTO days can be rolled over.

Overwriting policy.txt


It seems that `RetrievalQA` is no longer the standard way to build a RAG chain in the latest versions of LangChain. The recommended approach is to use LangChain Expression Language (LCEL).

Here is the updated code using LCEL to create a retrieval chain.

In [ ]:
import os
# from langchain_openai import ChatOpenAI # Comment out or remove this line
from langchain_google_genai import ChatGoogleGenerativeAI # Import the Gemini integration
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from google.colab import userdata # Import userdata to access Colab Secrets

# --- 1. SET UP THE LLM ---
# Use the Gemini API key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

if "GOOGLE_API_KEY" not in os.environ:
    print("Error: GOOGLE_API_KEY not found. Please set it in Colab Secrets (🔑).")
else:
    # Use the Gemini model
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0) # Changed model to gemini-1.5-flash

    # --- 2. LOAD YOUR DATA ---
    loader = TextLoader("policy.txt")
    documents = loader.load()

    # --- 3. CHUNK THE DATA ---
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    docs = text_splitter.split_documents(documents)

    # --- 4. CREATE EMBEDDINGS & VECTOR STORE (Still Free) ---
    # We still use the free, local model for embeddings.
    print("Loading local embedding model...")
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    print("Creating vector store...")
    vector_store = FAISS.from_documents(docs, embeddings)
    print("Vector store created.")

    # --- 5. CREATE THE RAG CHAIN USING LCEL ---
    retriever = vector_store.as_retriever()

    # Define the prompt template
    template = """Answer the question based only on the following context:
    {context}

    Question: {question}
    """
    prompt = ChatPromptTemplate.from_template(template)

    # Create the RAG chain using LCEL
    rag_chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )


    # --- 6. ASK QUESTIONS! ---
    print("\n--- Smart-Policy Bot is Ready! ---")
    print("Ask questions about the company policy (type 'exit' to quit).")

    while True:
        query = input("\nYour Question: ")
        if query.lower() == "exit":
            break

        # Pass the query to the RAG chain
        result = rag_chain.invoke(query)

        # Print the answer
        print("Answer:", result)

Loading local embedding model...
Creating vector store...
Vector store created.

--- Smart-Policy Bot is Ready! ---
Ask questions about the company policy (type 'exit' to quit).


In [21]:
!pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.9 MB/s  0:00:00
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langchain-google-genai]
